# Brazil Property Prices: Data Understanding

### Objective

This notebook examines, cleans, and integrates two Brazilian real estate datasets provided through the WorldQuant University Applied Data Science Lab course materials.

The analysis focuses on:

- understanding the structure and variables in each dataset
- identifying missing values and data-quality issues
- standardizing variables across the datasets
- engineering compatible features where necessary
- combining the cleaned datasets into a unified analytical dataset

The resulting dataset will be used for the subsequent analysis in this project.

The cleaned dataset is also structured so that it can support future statistical or machine-learning analysis, including Linear Regression. **Linear Regression is not developed or evaluated in this project.**

In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd

from scipy.spatial.distance import cdist

pd.set_option("display.max_columns", None)

# Define the path to our first data file
path_1 = "../data/raw/brasil-real-estate-1.csv"
print(f"We will load data from: '{path_1}'")

We will load data from: '../data/raw/brasil-real-estate-1.csv'


In [2]:
# Preview the dataset
(pd.read_csv(path_1).head())

,property_type,place_with_parent_names,region,lat-lon,area_m2,price_usd
0,apartment,|Brasil|Alagoas|Maceió|,Northeast,"-9.6443051,-35.7088142",110.0,"$187,230.85"
1,apartment,|Brasil|Alagoas|Maceió|,Northeast,"-9.6430934,-35.70484",65.0,"$81,133.37"
2,house,|Brasil|Alagoas|Maceió|,Northeast,"-9.6227033,-35.7297953",211.0,"$154,465.45"
3,apartment,|Brasil|Alagoas|Maceió|,Northeast,"-9.622837,-35.719556",99.0,"$146,013.20"
4,apartment,|Brasil|Alagoas|Maceió|,Northeast,"-9.654955,-35.700227",55.0,"$101,416.71"


In [3]:
# Preview the end of the dataset
(pd.read_csv(path_1).tail())

,property_type,place_with_parent_names,region,lat-lon,area_m2,price_usd
12829,apartment,|Brasil|Pernambuco|Recife|,Northeast,"-8.056418,-34.909309",91.0,"$174,748.79"
12830,apartment,|Brasil|Pernambuco|Recife|,Northeast,"-8.1373477,-34.909181",115.0,"$115,459.02"
12831,apartment,|Brasil|Pernambuco|Recife|Boa Viagem|,Northeast,"-8.1136717,-34.896252",76.0,"$137,302.62"
12832,apartment,|Brasil|Pernambuco|Recife|Boa Viagem|,Northeast,NaN,130.0,"$234,038.56"
12833,apartment,|Brasil|Pernambuco|Recife|Boa Viagem|,Northeast,"-8.0578381,-34.882897",99.0,"$168,507.77"


## Initial Data Assessment — Dataset 1

The first dataset is examined to understand its structure, variable types, missing values, and potential data-quality issues before transformation.

The initial assessment focuses on identifying:

- the available variables and their data types
- missing observations
- inconsistent representations of numeric values
- geographic information stored in combined fields
- the suitability of the available variables for subsequent analysis

In [4]:
# Check dataset schema and non-null counts
(pd.read_csv(path_1).info())

<class 'pandas.DataFrame'>
RangeIndex: 12834 entries, 0 to 12833
Data columns (total 6 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   property_type            12834 non-null  str    
 1   place_with_parent_names  12834 non-null  str    
 2   region                   12834 non-null  str    
 3   lat-lon                  11551 non-null  str    
 4   area_m2                  12834 non-null  float64
 5   price_usd                12834 non-null  str    
dtypes: float64(1), str(5)
memory usage: 601.7 KB


### Key Insights from `.info()`

* **Data Types:** The `price_usd` and `lat-lon` columns are stored as text strings rather than numeric values, which will prevent mathematical analysis and mapping until converted.
* **Missing Data:** The `lat-lon` column is missing coordinates for **1,283** rows (11,551 non-null out of 12,834).


In [5]:
# Quantify total missing values per column
(pd.read_csv(path_1).isnull().sum())

property_type                 0
place_with_parent_names       0
region                        0
lat-lon                    1283
area_m2                       0
price_usd                     0
dtype: int64

In [6]:
# Check missing coordinates by Property Type
(
    pd.read_csv(path_1)
    .assign(is_missing=lambda df: df["lat-lon"].isnull())
    .groupby("property_type")["is_missing"]
    .agg(["sum", "mean"])
)

,sum,mean
property_type,,
apartment,1068,0.099878
house,215,0.100420


In [7]:
# Check missing coordinates by Region
(
    pd.read_csv(path_1)
    .assign(is_missing=lambda df: df["lat-lon"].isnull())
    .groupby("region")["is_missing"]
    .agg(["sum", "mean"])
)

,sum,mean
region,,
Central-West,143,0.093464
North,36,0.106195
Northeast,457,0.100683
South,273,0.096912
Southeast,374,0.103630


## Data Quality Assessment — Dataset 1

The initial inspection identifies several issues that require treatment before the datasets can be combined.

### Numeric Representation

`price_usd` is stored as a string and requires conversion to a numeric representation before quantitative analysis.

### Geographic Representation

The `lat-lon` field combines latitude and longitude in a single text field. These coordinates need to be separated into numeric latitude and longitude variables.

### Missing Geographic Information

Some records do not contain coordinate information. These observations are retained unless a later analytical step requires valid coordinates.

### Missingness Assessment

Missing values were examined across available categorical groupings to determine whether missing observations appear concentrated within particular property types or geographic regions.

The grouped comparisons provide evidence about the distribution of missingness but do not establish a formal missing-data mechanism such as MCAR (Missing Completely at Random).

## Data Cleaning & Transformation Plan — Dataset 1

The first dataset will be transformed to produce consistent numeric and geographic variables while retaining observations where the available information remains analytically useful.

The main transformations are:

1. Convert `price_usd` from text to numeric values.
2. Split the combined `lat-lon` field into separate latitude and longitude variables.
3. Standardize the resulting schema for compatibility with the second dataset.
4. Retain missing observations where they do not prevent the required downstream analysis.

## Missing Data Strategy

Missing observations are not removed indiscriminately.

The decision to retain observations is based on whether the missing field can be addressed using information already available in the dataset and whether removing the observation would unnecessarily reduce the analytical population.

Where appropriate, missing values are handled through contextual feature engineering. Observations for which a reliable value cannot be derived are retained when possible and excluded only from analyses that require the missing variable.

This approach prioritizes preservation of useful observations while keeping the treatment of missing data explicit.

In [8]:
# Complete cleaning pipeline with fallback spatial imputation
df1 = (
    pd.read_csv(path_1)
    .assign(
        # Convert "$187,230.85" -> 187230.85 (Float)
        price_usd=lambda x: x["price_usd"]
            .str.replace("$", "", regex=False)
            .str.replace(",", "", regex=False)
            .astype(float),

        # Split text geo string into separate row float arrays
        lat=lambda x: x["lat-lon"]
            .str.split(",", expand=True)[0]
            .astype(float),
        lon=lambda x: x["lat-lon"]
            .str.split(",", expand=True)[1]
            .astype(float),

        # Extract cleam categorical columns
        # (index 2 is state, index 3 is city)
        state=lambda x: x["place_with_parent_names"]
            .str.split("|", expand=True)[2],
        city=lambda x: x["place_with_parent_names"]
            .str.split("|", expand=True)[3]
    )

    # Spatial Imputation using group-levele transform medians
    .assign(
        lat=lambda x: x["lat"]
            .fillna(
                x
                    .groupby("city")["lat"]
                    .transform("median")
            ),
        lon=lambda x: x["lon"]
            .fillna(
                x
                    .groupby("city")["lon"]
                    .transform("median")
            )
    )

    # Fallback Imputation: Fill any remaining edge
    # using state medians
    .assign(
        lat=lambda x: x["lat"]
            .fillna(
                x
                    .groupby("state")["lat"]
                    .transform("median")
            ),
        lon=lambda x: x["lon"]
            .fillna(
                x
                    .groupby("state")["lon"]
                    .transform("median")
            )
    )

    # Final structural cleanup
    .drop(columns=["lat-lon", "place_with_parent_names"])
)

# Verify the fully preserved machine learning matrix
print(f"Preseverd matrix shape: {df1.shape}")
print(f"'price_usd' dtype: {df1["price_usd"].dtype}")
print(f"'Lat' dtype: {df1["lat"].dtype}")
print(f"'Lon' dtype: {df1["lon"].dtype}")
print(f"'Lat' missing values: {df1["lat"].isnull().sum()}")
print(f"'Lon' missing values: {df1["lon"].isnull().sum()}")

Preseverd matrix shape: (12834, 8)
'price_usd' dtype: float64
'Lat' dtype: float64
'Lon' dtype: float64
'Lat' missing values: 0
'Lon' missing values: 0


In [9]:
df1.info()

<class 'pandas.DataFrame'>
RangeIndex: 12834 entries, 0 to 12833
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   property_type  12834 non-null  str    
 1   region         12834 non-null  str    
 2   area_m2        12834 non-null  float64
 3   price_usd      12834 non-null  float64
 4   lat            12834 non-null  float64
 5   lon            12834 non-null  float64
 6   state          12834 non-null  str    
 7   city           12834 non-null  str    
dtypes: float64(4), str(4)
memory usage: 802.3 KB


## Initial Data Assessment — Dataset 2

The second dataset is assessed independently before integration with the first cleaned dataset.

The assessment focuses on:

- schema differences between the datasets
- variable types and representations
- geographic information
- currency differences
- missing observations
- compatibility with the common analytical schema

In [10]:
# Define the path to our second data file
path_2 = "../data/raw/brasil-real-estate-2.csv"
print(f"We will load data from: '{path_2}'")

We will load data from: '../data/raw/brasil-real-estate-2.csv'


In [11]:
# Ispection the data
(pd.read_csv(path_2).head())

,property_type,state,region,lat,lon,area_m2,price_brl
0,apartment,Pernambuco,Northeast,-8.134204,-34.906326,72.0,414222.98
1,apartment,Pernambuco,Northeast,-8.126664,-34.903924,136.0,848408.53
2,apartment,Pernambuco,Northeast,-8.125550,-34.907601,75.0,299438.28
3,apartment,Pernambuco,Northeast,-8.120249,-34.895920,187.0,848408.53
4,apartment,Pernambuco,Northeast,-8.142666,-34.906906,80.0,464129.36


In [12]:
(pd.read_csv(path_2).tail())

,property_type,state,region,lat,lon,area_m2,price_brl
12828,house,São Paulo,Southeast,-23.587495,-46.559401,250.0,429194.89
12829,apartment,São Paulo,Southeast,-23.522029,-46.189290,55.0,252398.80
12830,apartment,São Paulo,Southeast,-23.526443,-46.529182,57.0,319400.84
12831,house,Tocantins,North,-8.848399,-48.511164,NaN,529007.65
12832,apartment,Tocantins,North,-10.249091,-48.324286,70.0,289457.01


In [13]:
(pd.read_csv(path_2).shape)

(12833, 7)

In [14]:
(pd.read_csv(path_2).info())

<class 'pandas.DataFrame'>
RangeIndex: 12833 entries, 0 to 12832
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   property_type  12833 non-null  str    
 1   state          12833 non-null  str    
 2   region         12833 non-null  str    
 3   lat            12833 non-null  float64
 4   lon            12833 non-null  float64
 5   area_m2        11293 non-null  float64
 6   price_brl      12833 non-null  float64
dtypes: float64(4), str(3)
memory usage: 701.9 KB


In [15]:
(pd.read_csv(path_2).isnull().sum())

property_type       0
state               0
region              0
lat                 0
lon                 0
area_m2          1540
price_brl           0
dtype: int64

## Initial Data Assessment & Data Anomalies — Dataset 2

The second dataset differs from the first dataset in several important ways.

### 1. Schema Differences

Dataset 2 does not contain a `city` field. It contains `state` and `region`, together with latitude and longitude coordinates.

### 2. Geographic Representation

Latitude and longitude are already provided as separate numeric variables in Dataset 2.

### 3. Currency Representation

Property prices are recorded in Brazilian Real (`price_brl`), whereas the first dataset uses `price_usd`. A common currency representation is therefore required before the datasets can be combined.

### 4. Missing Property Area

`area_m2` contains 1,540 missing observations. Because property area is relevant to the subsequent price-per-square-meter analysis, the missing values require explicit treatment rather than indiscriminate row removal.

In [16]:
# Check missing area_m2 by Property Type
(
    pd.read_csv(path_2)
    .assign(is_missing=lambda df: df["area_m2"].isnull())
    .groupby("property_type")["is_missing"]
    .agg(["sum", "mean"])
)

,sum,mean
property_type,,
apartment,1219,0.121209
house,321,0.115634


In [17]:
# Check missing area_m2 by Region
(
    pd.read_csv(path_2)
    .assign(is_missing=lambda df: df["area_m2"].isnull())
    .groupby("region")["is_missing"]
    .agg(["sum", "mean"])
)

,sum,mean
region,,
North,1,0.142857
Northeast,114,0.138015
South,723,0.120500
Southeast,702,0.117000


## Missing Data Distribution Analysis — Dataset 2

The missing `area_m2` observations were examined across property type and geographic region.

The observed missingness rates are relatively similar across the inspected groups, with no strong concentration apparent in the comparisons performed.

These checks describe the observed distribution of missing values but do not establish that the missingness is Missing Completely at Random (MCAR).

Because `area_m2` is an important analytical variable and the missing observations can be addressed using available contextual information, the records will be retained and the missing values will be handled through grouped median imputation.

## Data Cleaning & Transformation Plan — Dataset 2

Dataset 2 requires several transformations before it can be integrated with Dataset 1.

The cleaning process will:

1. Normalize property prices to the common `price_usd` representation.
2. Address missing `area_m2` values using contextual grouped medians.
3. Derive `city` information from geographic proximity to the available reference data.
4. Remove the original `price_brl` field after currency normalization.
5. Align the final column structure with Dataset 1.

### Currency Normalization

Dataset 2 records property prices in Brazilian Real (`price_brl`), while Dataset 1 uses `price_usd`.

To make the datasets structurally comparable, `price_brl` is converted to `price_usd` using the 2014 historical average exchange-rate benchmark supplied for this analysis.

The original `price_brl` field is subsequently removed after the normalized price variable has been created.

### Contextual Imputation of Property Area

The missing `area_m2` values are retained rather than removing the affected observations.

A grouped median based on `property_type` and `region` is used to provide a contextual estimate for missing property-area values.

This approach preserves the observations while allowing the imputed values to reflect differences between property types and broad geographic regions.

The resulting dataset is subsequently checked to confirm that the transformation does not leave missing values in the analytical fields required for integration.

### Geographic Resolution

Dataset 2 does not provide a city variable, but it contains latitude and longitude coordinates.

A geographic reference table is therefore created from Dataset 1 using available coordinate and city information.

Each Dataset 2 coordinate is matched to the nearest coordinate in this reference table using pairwise geographic distance calculations. The corresponding city value is then assigned to Dataset 2.

The resulting `city` variable is therefore a derived feature based on geographic proximity rather than an original field in Dataset 2.

In [52]:
# Create a clean geographic refernce table from first dataset
geo_reference = (
    df1[["lat", "lon", "city"]]
        .dropna()
        .drop_duplicates()
)

# Run the complete unified data cleaning and spatial
# resolution pipeline
df2 = (
    pd.read_csv(path_2)
    # Convert BRL to USD using 2014 benchmark and fill
    # area metrics using grouped medians
    .assign(
        price_usd=lambda x: x["price_brl"] / 2.3533,
        area_m2=lambda x: x["area_m2"]
            .fillna(
                x
                    .groupby(["property_type", "region"])["area_m2"]
                    .transform("median")
            )
    ).round(2)

    # Custom spatial resolution pipeline using .pipe()
    .pipe(lambda x: x.assign(
        city=geo_reference["city"].iloc[
            cdist(
                x[["lat", "lon"]]
                    .values, geo_reference[["lat", "lon"]]
                    .values
            ).argmin(axis=1)
        ].values
    ))

    # Dimensionality cleanup $ absolute column alignment with df1
    .drop(columns=["price_brl"])[df1.columns]
)

In [53]:
df2.head()

,property_type,region,area_m2,price_usd,lat,lon,state,city
0,apartment,Northeast,72.0,176017.92,-8.13,-34.91,Pernambuco,Recife
1,apartment,Northeast,136.0,360518.65,-8.13,-34.90,Pernambuco,Recife
2,apartment,Northeast,75.0,127241.86,-8.13,-34.91,Pernambuco,Recife
3,apartment,Northeast,187.0,360518.65,-8.12,-34.90,Pernambuco,Recife
4,apartment,Northeast,80.0,197224.90,-8.14,-34.91,Pernambuco,Recife


In [54]:
df2.tail()

,property_type,region,area_m2,price_usd,lat,lon,state,city
12828,house,Southeast,250.0,182380.02,-23.59,-46.56,São Paulo,Santo André
12829,apartment,Southeast,55.0,107253.13,-23.52,-46.19,São Paulo,Santo André
12830,apartment,Southeast,57.0,135724.66,-23.53,-46.53,São Paulo,Santo André
12831,house,North,175.0,224793.97,-8.85,-48.51,Tocantins,Imperatriz
12832,apartment,North,70.0,123000.47,-10.25,-48.32,Tocantins,Barreiras


In [55]:
df2.shape

(12833, 8)

In [56]:
df2.isnull().sum()

property_type    0
region           0
area_m2          0
price_usd        0
lat              0
lon              0
state            0
city             0
dtype: int64

In [57]:
df2.info()

<class 'pandas.DataFrame'>
RangeIndex: 12833 entries, 0 to 12832
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   property_type  12833 non-null  str    
 1   region         12833 non-null  str    
 2   area_m2        12833 non-null  float64
 3   price_usd      12833 non-null  float64
 4   lat            12833 non-null  float64
 5   lon            12833 non-null  float64
 6   state          12833 non-null  str    
 7   city           12833 non-null  str    
dtypes: float64(4), str(4)
memory usage: 802.2 KB


## Dataset 2 Validation

After transformation, Dataset 2 is checked for:

- expected row and column dimensions
- missing values
- data types
- column compatibility with Dataset 1
- successful creation of the derived geographic field

The validation confirms that the transformed dataset can be aligned with Dataset 1 for vertical concatenation.

In [58]:
# Check if all column sets are identical
columns_match = set(df1) == set(df2)

if columns_match:
    print("✔ All DataFrames have the same columns.")
else:
    print("✗ Column mismatch detected!")
    ## See the difference
    print("df1 vs df2 diff:", set(df1.columns) ^ set(df2.columns))

✔ All DataFrames have the same columns.


## Dataset Unification

Both datasets have now been transformed to a common analytical structure.

The cleaned datasets are vertically concatenated into a single dataset, preserving all observations from both sources.

The combined dataset is then validated for:

- expected dimensions
- consistent column structure
- missing values
- compatible data types

The resulting dataset provides the analytical foundation for the remaining notebooks in this project.

In [61]:
# Vertically stack the two cleaned dataframes
df = pd.concat([df1, df2], ignore_index=True)

# Verify the dimensions, column integrity, and total
# row structure
print(f"\nMaster Matrix Dimension Summary: {df.shape}\n")
df.info()


Master Matrix Dimension Summary: (25667, 8)

<class 'pandas.DataFrame'>
RangeIndex: 25667 entries, 0 to 25666
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   property_type  25667 non-null  str    
 1   region         25667 non-null  str    
 2   area_m2        25667 non-null  float64
 3   price_usd      25667 non-null  float64
 4   lat            25667 non-null  float64
 5   lon            25667 non-null  float64
 6   state          25667 non-null  str    
 7   city           25667 non-null  str    
dtypes: float64(4), str(4)
memory usage: 1.6 MB


## Export Cleaned Dataset

The validated combined dataset is exported to the processed-data directory for use in the subsequent analysis notebooks.

This creates a reproducible separation between the original source data and the cleaned analytical dataset.

In [62]:
# Export the cleaned, combined dataset to a CSV file
df.to_csv(
    "../data/processed/brasil-real-estate-combined-clean.csv",
    index=False
)

## Conclusion

The data-understanding stage established a consistent analytical foundation from the two Brazilian real estate datasets.

The assessment identified differences in variable structure, price representation, geographic fields, and missing property-area observations. These issues were addressed through targeted data cleaning and feature engineering while retaining observations where reasonable contextual values could be derived.

Both datasets were transformed to a common schema, property prices were standardized to `price_usd`, missing `area_m2` values in Dataset 2 were addressed using grouped medians, and city information was derived from geographic proximity where it was not originally available. The resulting datasets were validated and successfully combined into a single analytical dataset containing 25,667 observations and eight variables.

The cleaned dataset provides a consistent foundation for the subsequent notebooks, where housing characteristics, property-price relationships, price per square meter, and location effects will be examined.